In [ ]:
%load_ext autoreload
%autoreload 2
DIANNE = ".."
sys.path.insert(0, f"{DIANNE}/dianne-core")
sys.path.insert(0, f"{DIANNE}/dianne-utils")
sys.path.insert(0, f"{DIANNE}/dianne-viewer")
import dianne_core
import dianne_utils as dianne
import dianne_viewer as viewer

import numpy as np
import pandas as pd

dianne.setNotebookWidth(100)

In [ ]:
# Download the demo data from Zenodo and unzip it to the data folder
dataPath = '../../data/'
dianne.downloadZIPFromZenodo(dataPath, 'https://zenodo.org/records/19225051/files/', 'RMS2194.oid0.zip')
dianne.downloadFromZenodo(dataPath + 'sarcoma', 'https://zenodo.org/api/records/19240981/files-archive')

In [ ]:
# Load the features and metadata for the sarcoma dataset
df = pd.read_csv(dataPath + '/sarcoma/wsi-data-ctranspath-1.csv.gz', index_col=[0, 1], header=[0, 1])
df.index = pd.MultiIndex.from_arrays([df.index.get_level_values('Tissue ID'), ['Sarcoma'] * len(df)], names=['Tissue ID', 'Dataset'])
df.columns = pd.MultiIndex.from_arrays([df.columns.get_level_values(0).astype(float), df.columns.get_level_values(1)])
df_meta = pd.read_csv(dataPath + '/sarcoma/wsi-metadata-ctranspath-1.csv.gz', index_col=0, header=0)

In [ ]:
# Create static annotations for training a classifier, excluding the test slide(s)
test_slides = ['RMS2194.oid0']
target_columns, target_class = 'Specimen Type', 'Tumor'
positive = df_meta.index[df_meta[target_columns] == target_class].difference(set(test_slides))
negative = df_meta.index[df_meta[target_columns] != target_class].difference(set(test_slides))
static_annotations = {}
static_annotations.update({(p, 'Sarcoma'): 'positive' for p in positive})
static_annotations.update({(p, 'Sarcoma'): 'negative' for p in negative})
print('Number of positive:', positive.shape, 'Number of negative:', negative.shape)

In [ ]:
# Train a classifier using the static annotations and the features in the dataframe
clf = dianne_core.trainClassifier(static_annotations, df, alpha=0.8, seed=0, augFunc=dianne_core.PCMA)

In [ ]:
# Run inference on the test slide, excluded from training, and visualize the results
infSample = 'RMS2194.oid0'
ad, img = dianne_core.loadAd(dataPath + '/RMS2194.oid0/', fname='/features/false-1-ctranspath_features.tsv.gz')
qs = np.linspace(0.05, 0.95, 10, endpoint=True)
ts, mpp = 112., 0.25
x, y, p = dianne.inferProb(ad, clf, qs, tsize=ts/mpp, R=2, erode=False)
xi, yi, pi = dianne.interpolatePoints(x, y, p, multiplier=16)
dianne.showProbImg(xi, yi, pi, f=2, figsize=(3, 3), ts=ts, mpp=mpp, title=infSample, invert=False)

In [ ]:
# Create probability masks, extract contours and visualize them on the H&E image
downsampled_map, fshape = dianne.makeProbMask(ad, img, x, y, p, ts=ts, mpp=mpp, downfactor=16, verbose=True)
geojson = dianne.extractContoursForQuPath(downsampled_map, fshape, cutoff=0.5, min_area=10**6, downfactor=16, sigma=100)
dianne.viewContoursOnImage(img, geojson, fshape, level=2, figsize=(12, 12), linewidth=1)